# Behind the Curve? Evaluating Bank of England and ECB Interest Rate Decisions Against the Taylor Rule, 2019–2024

## 1. Introduction

Between 2021 and 2023, the United Kingdom and the euro area experienced their highest inflation in around four decades, with UK CPI inflation peaking at 11.1% in October 2022. Both the Bank of England (BoE) and the European Central Bank (ECB) responded by raising interest rates, but both have been criticised for acting too slowly.

This project asks: **did the BoE and the ECB raise interest rates later and by less than a standard monetary policy rule recommended, and if so, does that reflect a policy error or a limitation of the rule itself?**

To answer this, I compare each central bank's actual policy rate with the rate implied by the Taylor rule (Taylor, 1993), month by month, from January 2019 to December 2024, using official data from the ONS, the Bank of England and the ECB.

**Contents**
1. Introduction
2. Data
3. Method: the Taylor rule
4. Results
5. Robustness check
6. Discussion
7. Limitations and further research

## 2. Data

In [ ]:
import io
import requests
import pandas as pd
import matplotlib.pyplot as plt

START = "2019-01"   # a pre-pandemic baseline year
END = "2024-12"     # covers the full tightening cycle and the first rate cuts

# Some official data portals reject requests without a user agent
HEADERS = {"User-Agent": "Mozilla/5.0 (economics research project)"}

def download(url):
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r

Four monthly series are used, all downloaded directly from official sources:

| Series | Source | Code |
|---|---|---|
| UK CPI inflation, annual rate (%) | Office for National Statistics | D7G7 |
| Bank Rate (%) | Bank of England | IUDBEDR |
| Euro area HICP inflation, annual rate (%) | European Central Bank | ICP.M.U2.N.000000.4.ANR |
| ECB deposit facility rate (%) | European Central Bank | FM.B.U2.EUR.4F.KR.DFR.LEV |

For the euro area, I use the **deposit facility rate** rather than the main refinancing rate. Because of the large excess liquidity in the banking system after 2015, the deposit rate was the ECB's effective policy rate throughout this period and is the closest equivalent to Bank Rate.

Policy rates change on specific decision dates, so each series is converted to the rate in force at the end of each month. Where a source records only the dates on which the rate changed, intervening months are filled with the most recent rate.

In [ ]:
def uk_cpi():
    # ONS publishes each time series as JSON; "months" holds the monthly observations
    data = download("https://www.ons.gov.uk/economy/inflationandpriceindices/timeseries/d7g7/mm23/data").json()
    df = pd.DataFrame(data["months"])
    df["month"] = pd.to_datetime(df["date"], format="%Y %b").dt.to_period("M")
    return df.set_index("month")["value"].astype(float)

def boe_bank_rate():
    url = ("https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp"
           "?csv.x=yes&Datefrom=01/Jan/2018&Dateto=31/Dec/2024"
           "&SeriesCodes=IUDBEDR&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N")
    df = pd.read_csv(io.StringIO(download(url).text))
    df["date"] = pd.to_datetime(df["DATE"], format="%d %b %Y")
    daily = df.set_index("date")["IUDBEDR"].astype(float).sort_index()
    return every_month(daily.groupby(daily.index.to_period("M")).last())

def every_month(rate):
    # A policy rate stays in force until the next decision. Some sources record only the
    # dates of changes, so each month is filled with the most recent rate.
    months = pd.period_range("2018-01", "2024-12", freq="M")
    return rate.reindex(rate.index.union(months)).ffill().loc[months]

def ecb_series(flow, key, start="2018-01-01"):
    url = (f"https://data-api.ecb.europa.eu/service/data/{flow}/{key}"
           f"?startPeriod={start}&endPeriod=2024-12-31&format=csvdata")
    df = pd.read_csv(io.StringIO(download(url).text))
    df["date"] = pd.to_datetime(df["TIME_PERIOD"])
    s = df.set_index("date")["OBS_VALUE"].astype(float).sort_index()
    return s.groupby(s.index.to_period("M")).last()

def euro_hicp():
    return ecb_series("ICP", "M.U2.N.000000.4.ANR")   # euro area, all items, annual rate

def ecb_deposit_rate():
    last_error = None
    for key in ["B.U2.EUR.4F.KR.DFR.LEV", "D.U2.EUR.4F.KR.DFR.LEV"]:  # both frequency codes are in use
        try:
            # start in 2010 to capture the last change before the sample (March 2016)
            return every_month(ecb_series("FM", key, start="2010-01-01"))
        except Exception as e:
            last_error = e
    raise last_error

As a fallback, if either policy rate cannot be downloaded, the series is reconstructed from the official monetary policy decision dates published by each central bank. The inflation series have no fallback.

In [ ]:
# (date the new rate took effect, new rate in %)
BOE_DECISIONS = [("2018-08-02", 0.75), ("2020-03-11", 0.25), ("2020-03-19", 0.10),
    ("2021-12-16", 0.25), ("2022-02-03", 0.50), ("2022-03-17", 0.75), ("2022-05-05", 1.00),
    ("2022-06-16", 1.25), ("2022-08-04", 1.75), ("2022-09-22", 2.25), ("2022-11-03", 3.00),
    ("2022-12-15", 3.50), ("2023-02-02", 4.00), ("2023-03-23", 4.25), ("2023-05-11", 4.50),
    ("2023-06-22", 5.00), ("2023-08-03", 5.25), ("2024-08-01", 5.00), ("2024-11-07", 4.75)]
ECB_DFR_DECISIONS = [("2016-03-16", -0.40), ("2019-09-18", -0.50), ("2022-07-27", 0.00),
    ("2022-09-14", 0.75), ("2022-11-02", 1.50), ("2022-12-21", 2.00), ("2023-02-08", 2.50),
    ("2023-03-22", 3.00), ("2023-05-10", 3.25), ("2023-06-21", 3.50), ("2023-08-02", 3.75),
    ("2023-09-20", 4.00), ("2024-06-12", 3.75), ("2024-09-18", 3.50), ("2024-10-23", 3.25),
    ("2024-12-18", 3.00)]

def rate_from_decisions(decisions):
    s = pd.Series({pd.Timestamp(d): r for d, r in decisions})
    months = pd.period_range(START, END, freq="M")
    return pd.Series([s[s.index <= m.end_time].iloc[-1] for m in months], index=months)

def get(name, fetch, backup=None):
    try:
        s = fetch()
        print(f"✓ {name}: downloaded")
        return s
    except Exception as e:
        if backup is None:
            raise RuntimeError(f"Could not download {name} ({e}). Please try again later.")
        print(f"! {name}: download failed ({type(e).__name__}); reconstructed from official decision dates")
        return rate_from_decisions(backup)

uk = pd.DataFrame({"inflation": get("UK CPI inflation", uk_cpi),
                   "actual_rate": get("Bank Rate", boe_bank_rate, BOE_DECISIONS)})
ea = pd.DataFrame({"inflation": get("Euro area HICP inflation", euro_hicp),
                   "actual_rate": get("ECB deposit rate", ecb_deposit_rate, ECB_DFR_DECISIONS)})

uk = uk.loc[START:END].dropna()
ea = ea.loc[START:END].dropna()
print(f"\nUK: {len(uk)} months, euro area: {len(ea)} months")
uk.tail()

The data match published figures: UK CPI inflation peaks at 11.1% in October 2022, and Bank Rate reaches 5.25% in August 2023.

## 3. Method: the Taylor rule

Taylor (1993) showed that US interest rate decisions could be closely described by a simple rule:

$$ i = r^* + \pi + 0.5\,(\pi - \pi^*) + 0.5\,y $$

| Symbol | Meaning | Value used |
|---|---|---|
| $i$ | recommended nominal policy rate | calculated |
| $r^*$ | neutral real interest rate | 2% (Taylor's original value); 0.5% in the robustness check |
| $\pi$ | annual inflation | from the data |
| $\pi^*$ | inflation target | 2% (both central banks) |
| $y$ | output gap (% deviation of GDP from potential) | 0 |

The coefficient on inflation is effectively 1.5: when inflation rises by one percentage point, the rule recommends raising the nominal rate by 1.5 points. This means the **real** interest rate ($i - \pi$) rises, which is necessary to reduce demand and bring inflation back to target. This is known as the *Taylor principle*.

**Simplifying assumption.** The output gap cannot be observed directly, and estimates during the pandemic were unusually uncertain and frequently revised. I therefore set $y = 0$, so the rule responds to inflation alone. The implications are discussed in Section 7.

In [ ]:
TARGET = 2.0

def taylor(inflation, neutral_real_rate=2.0, output_gap=0.0):
    return neutral_real_rate + inflation + 0.5 * (inflation - TARGET) + 0.5 * output_gap

for df in (uk, ea):
    df["taylor"] = taylor(df["inflation"])                                   # baseline: r* = 2%
    df["taylor_low_rstar"] = taylor(df["inflation"], neutral_real_rate=0.5)  # robustness: r* = 0.5%
    df["real_rate"] = df["actual_rate"] - df["inflation"]                    # ex-post real policy rate
    df["gap"] = df["actual_rate"] - df["taylor"]   # negative: policy looser than the rule implies

uk.loc["2022-01":"2022-12"].round(2)

## 4. Results

### 4.1 Actual policy rates against the Taylor rule

Figure 1 plots inflation, the Taylor rule recommendation and the actual policy rate for each economy on a common scale. The shaded area shows the gap between the recommended and the actual rate.

In [ ]:
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8a85"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True, sharey=True)
for ax, df, name, bank in [(axes[0], uk, "United Kingdom", "Bank Rate"),
                           (axes[1], ea, "Euro area", "ECB deposit rate")]:
    x = df.index.to_timestamp()
    ax.fill_between(x, df["actual_rate"], df["taylor"], where=df["taylor"] > df["actual_rate"],
                    color=ORANGE, alpha=0.12, linewidth=0)
    ax.plot(x, df["inflation"], color=GREY, linewidth=1.5, label="Inflation")
    ax.plot(x, df["taylor"], color=ORANGE, linewidth=2, linestyle="--", label="Taylor rule recommendation")
    ax.plot(x, df["actual_rate"], color=BLUE, linewidth=2.5, label=f"Actual rate ({bank})")
    ax.axhline(0, color="#cfcfca", linewidth=1)
    ax.grid(axis="y", color="#e6e6e1", linewidth=0.8)
    ax.set_title(name, loc="left", fontweight="bold")
    ax.set_ylabel("%")
    # direct end-of-line labels, nudged apart where lines end close together
    ends = sorted([(df[c].iloc[-1], t) for c, t in [("actual_rate", "Actual rate"),
                   ("taylor", "Taylor rule"), ("inflation", "Inflation")]])
    placed = []
    for y, text in ends:
        y_label = max(y, placed[-1] + 1.1) if placed else y
        placed.append(y_label)
        ax.annotate(text, (x[-1], y), xytext=(x[-1] + pd.Timedelta(days=20), y_label),
                    color="#3d3d3a", fontsize=9, va="center")

axes[0].legend(loc="upper left", frameon=False)
fig.suptitle("Figure 1. Policy rates vs the Taylor rule, 2019–2024", x=0.06, ha="left", fontsize=15, fontweight="bold")
fig.text(0.06, 0.005, "Sources: ONS, Bank of England, ECB. Taylor rule with neutral real rate 2%, "
         "inflation target 2%, output gap 0.", fontsize=8.5, color="#5c5c58")
fig.tight_layout(rect=(0, 0.02, 0.95, 0.97))
fig.savefig("taylor_rule_chart.png", dpi=200, bbox_inches="tight")
plt.show()

### 4.2 Summary measures

To make the comparison precise, I calculate the following for each economy:

- **Peak inflation** and the month in which it occurred.
- **Rule signal**: the first month from 2021 onwards (excluding the pandemic shock of 2020) in which the Taylor rule recommendation exceeded its 2019 average by more than one percentage point.
- **First rate rise**: the first month from 2021 in which the policy rate increased.
- **Lag**: the number of months between the rule signal and the first rate rise.
- **Maximum shortfall**: the largest gap between the recommended and the actual rate.
- **Persistence**: the number of months from 2021 in which the actual rate was more than three points below the recommendation.
- **Lowest real policy rate**: the policy rate minus inflation at its most negative.

In [ ]:
def months_between(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)

def summarise(df, name):
    after = df.loc["2021-01":]
    peak_month = df["inflation"].idxmax()
    normal = df.loc["2019", "taylor"].mean()
    rule_month = after.index[after["taylor"] > normal + 1][0]
    first_hike = after.index[after["actual_rate"].diff() > 0][0]
    worst_month = df["gap"].idxmin()
    return {
        "Economy": name,
        "Rule's normal level, 2019 (%)": round(normal, 1),
        "Peak inflation (%)": round(df["inflation"].max(), 1),
        "Peak inflation month": str(peak_month),
        "Rule first called for a rise": str(rule_month),
        "First actual rate rise": str(first_hike),
        "Lag (months)": months_between(rule_month, first_hike),
        "Biggest shortfall (points)": round(-df["gap"].min(), 1),
        "Biggest shortfall month": str(worst_month),
        "Rule recommended then (%)": round(df.loc[worst_month, "taylor"], 1),
        "Actual rate then (%)": round(df.loc[worst_month, "actual_rate"], 2),
        "Months > 3 points below rule": int((after["gap"] < -3).sum()),
        "Months from first rise to inflation peak": months_between(first_hike, peak_month),
        "Lowest real policy rate (%)": round(df["real_rate"].min(), 1),
        "Lowest real rate month": str(df["real_rate"].idxmin()),
    }

results = pd.DataFrame([summarise(uk, "UK"), summarise(ea, "Euro area")]).set_index("Economy").T
results

## 5. Robustness check

The neutral real rate $r^*$ is unobservable, and many estimates suggest it fell well below Taylor's original 2% after the 2008 financial crisis. Re-estimating the rule with $r^* = 0.5\%$ lowers every recommendation by 1.5 percentage points. If the shortfall remains large under this assumption, the main conclusion does not depend on the choice of $r^*$.

In [ ]:
for df, name in [(uk, "UK"), (ea, "Euro area")]:
    shortfall = (df["taylor_low_rstar"] - df["actual_rate"]).max()
    when = (df["taylor_low_rstar"] - df["actual_rate"]).idxmax()
    print(f"{name}: with r* = 0.5%, the maximum shortfall is {shortfall:.1f} percentage points ({when})")

## 6. Discussion

**Both central banks set rates far below the Taylor rule recommendation.** At the height of the inflation surge, the rule recommended policy rates of roughly 16–18%, while Bank Rate and the ECB deposit rate were still in low single figures. Real policy rates were deeply negative: in October 2022, with UK inflation at 11.1% and Bank Rate at 2.25%, the real rate was approximately −9%. Even after a sequence of rate rises, borrowing was therefore cheaper in real terms than at any point in recent decades. The robustness check shows that this conclusion holds with a lower neutral rate.

**The Bank of England moved earlier than the ECB.** The BoE began raising rates in December 2021, while the ECB did not raise its deposit rate until July 2022. However, inflation in both economies continued to rise for months after the first increases, and the gap between the recommended and actual rate widened during 2022 in both cases, because inflation rose faster than policy rates.

**Was this a policy error, or does the rule mislead here?** The Taylor rule treats all inflation as a signal of excess demand. Much of the 2022 inflation, however, was caused by a supply shock: Russia's invasion of Ukraine sharply reduced the supply of gas to Europe, driving up energy and food prices. Interest rates work by reducing demand; they cannot increase the supply of gas. Following the rule would have meant raising rates to around 17%, which would have more than tripled monthly repayments on a typical variable-rate mortgage and very likely caused a deep recession, without addressing the cause of the price rises. On this view, a policy rate well below the rule's recommendation was a reasonable response to a supply shock.

That said, the rule does capture a genuine risk. Both central banks initially described inflation as "transitory", and inflation broadened beyond energy into services and wages during 2022–23. Keeping real rates so negative for so long may have allowed high inflation to become embedded in expectations, which then required a longer period of tight policy to reverse. A plausible conclusion is therefore that the banks were right not to follow the rule mechanically, but were slower than ideal to respond once it became clear that inflation was spreading beyond energy.

## 7. Limitations and further research

- **Output gap.** Setting the output gap to zero means the rule ignores the state of the economy. Including official estimates (e.g. from the OBR or the IMF) would refine the recommendation.
- **Headline vs core inflation.** Using core inflation, which excludes energy and food, would test directly whether the rule's recommendation is driven by the supply shock.
- **Policy lags.** Monetary policy is thought to affect inflation with a lag of around 18–24 months, so a forward-looking rule based on inflation forecasts may be a fairer benchmark than current inflation.
- **One rate for many economies.** The ECB sets a single rate for 20 countries. Applying the rule to individual member states such as Germany would show how well a single rate suited each economy.
- **Other policy tools.** Both central banks also used quantitative tightening, which this analysis does not capture.

## References

- Taylor, J. B. (1993). Discretion versus policy rules in practice. *Carnegie-Rochester Conference Series on Public Policy*, 39, 195–214.
- Office for National Statistics. CPI annual rate (D7G7).
- Bank of England. Official Bank Rate (IUDBEDR).
- European Central Bank. HICP and key ECB interest rates, ECB Data Portal.